# Retail Decision Intelligence Platform

# Module 3 – Data Cleaning

## Objective

The objective of this notebook is to improve the quality of the retail dataset by identifying and handling duplicate records, missing values, cancelled transactions, invalid quantities, and incorrect prices.

The goal is to prepare a reliable dataset for exploratory analysis, customer segmentation, market basket analysis, forecasting, and dashboard development.

---

## Cleaning Strategy

The cleaning process will follow these steps:

1. Remove duplicate records
2. Investigate cancelled invoices
3. Handle missing descriptions
4. Handle missing customer IDs
5. Remove invalid quantities
6. Remove invalid prices
7. Create a cleaned dataset

In [44]:
import pandas as pd
import numpy as np 

#Display settings
pd.set_option("display.max_columns",None)
pd.set_option("display.width",1000)

#Load dataset
df = pd.read_excel("../data/raw/online_retail.xlsx")

print("Dataset Loaded successfully")

Dataset Loaded successfully


In [45]:
#Create a working copy

clean_df = df.copy()

print("copy created successfully")

copy created successfully


In [46]:
#Cancelled invoices start with 'C'

cancelled_orders = clean_df[clean_df["Invoice"].astype(str).str.startswith("C")]
print("Cancelled Transactions: ",len(cancelled_orders))

Cancelled Transactions:  10206


In [47]:
cancelled_orders.head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321.0,Australia
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom


# Investigation of Cancelled Transactions

The cancelled transactions were identified using invoices that begin with the letter **C**.

### Key Findings

- Cancelled invoices contain negative quantities.
- These records represent product returns or cancelled orders rather than data entry errors.
- They are valid business events and provide useful information about customer return behaviour.

### Cleaning Decision

For this project:

- Cancelled transactions will be excluded from the main sales analysis.
- They can be stored separately for future return analysis.

This ensures that revenue calculations and customer purchase analysis are based only on completed sales.

In [48]:
#Separate cancelled Transactions

returns_df = clean_df[clean_df["Invoice"].astype(str).str.startswith("C")].copy()
print("Number of Returned Transactions: ",len(returns_df))

Number of Returned Transactions:  10206


In [49]:
#Remove Cancelled Transactions

clean_df = clean_df[~clean_df["Invoice"].astype(str).str.startswith("C")]
print("Remaining Records: ", len(clean_df))

Remaining Records:  515255


In [50]:
clean_df["Invoice"].astype(str).str.startswith("C").sum()

np.int64(0)

In [51]:
cancelled_percentage = (len(returns_df) / len(df)) * 100
print(f"Cancelled Transactions Percentage: {cancelled_percentage: .2f}%")

Cancelled Transactions Percentage:  1.94%


In [52]:
#Check Negative Quantities

negative_quantity = clean_df[clean_df["Quantity"]<0]

print("Negative Quantity Records: ", len(negative_quantity))

Negative Quantity Records:  2121


In [53]:
#Check for zero Quantities

zero_quantity = clean_df[clean_df["Quantity"]==0]
print("Zero Quantity Records: ",len(zero_quantity))

Zero Quantity Records:  0


In [54]:
#Inspecting Negative Quantity records
negative_quantity.head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0,NaN,United Kingdom
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom


In [55]:
negative_quantity["Invoice"].head(20)

263      489464
283      489463
284      489467
470      489521
3114     489655
3162     489660
3168     489663
4296     489806
4538     489820
4566     489821
6556     489899
6576     489901
6911     490007
7205     490016
7559     490055
8553     490084
9308     490130
9667     490146
10508    490165
11973    490354
Name: Invoice, dtype: object

In [56]:
negative_quantity["Description"].value_counts().head(20)

Description
?                         42
damages                   39
damaged                   36
missing                   24
dotcom                     6
given away                 6
smashed                    5
checked                    5
MIA                        4
Damages                    4
ebay sales                 4
crushed                    4
No Stock                   3
Damaged                    3
counted                    3
check                      3
discoloured                2
damages?                   2
Dotcom                     2
damages, lost bits etc     2
Name: count, dtype: int64

In [57]:
negative_quantity["Price"].describe()

count    2121.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: Price, dtype: float64

# Investigation of Remaining Negative Quantities

After removing cancelled invoices, 2,121 records with negative quantities still remained.

Further investigation revealed that these records have the following characteristics:

- Invoice numbers do not begin with "C".
- Unit price is 0.
- Customer IDs are missing.
- Descriptions include terms such as:
  - damages
  - damaged
  - lost
  - missing
  - checked
  - given away
  - smashed
  - inventory adjustments

These records represent internal inventory operations rather than customer purchases.

## Cleaning Decision

Since the objective of this project is to analyse customer sales, these inventory adjustment records will be removed from the analytical dataset.

The decision ensures that sales metrics, customer segmentation, and forecasting are based only on genuine customer transactions.

In [58]:
#Removing Remaining Negative Quantities

rows_before = len(clean_df)
clean_df = clean_df[clean_df["Quantity"] > 0]
rows_after =len(clean_df)

print(f"Rows before : {rows_before:,}")
print(f"Rows after : {rows_after:,}")
print(f"Rows removed : {rows_before - rows_after:,}")

Rows before : 515,255
Rows after : 513,134
Rows removed : 2,121


In [59]:
(clean_df["Quantity"] < 0).sum()

np.int64(0)

In [60]:
#Zero and negative price
(clean_df["Price"] <= 0).sum()

np.int64(1569)

# Investigation of Zero-Priced Transactions

After removing cancelled transactions and inventory adjustments, the dataset still contains transactions where the unit price is zero.

These records will be investigated to determine whether they represent:

- Promotional giveaways
- Free samples
- Replacement products
- Internal stock transfers
- Data entry errors

Only after understanding their business context will a cleaning decision be made.

In [61]:
#Investigating Zero Price records

zero_price = clean_df[clean_df["Price"] <= 0]
zero_price.head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126.0,United Kingdom
5904,489861,DOT,DOTCOM POSTAGE,1,2009-12-02 14:50:00,0.0,NaN,United Kingdom
6378,489882,35751C,NaN,12,2009-12-02 16:22:00,0.0,NaN,United Kingdom
6555,489898,79323G,NaN,954,2009-12-03 09:40:00,0.0,NaN,United Kingdom
6581,489903,21166,NaN,48,2009-12-03 09:57:00,0.0,NaN,United Kingdom
6781,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.0,15658.0,United Kingdom
7204,490015,21982,NaN,467,2009-12-03 12:29:00,0.0,NaN,United Kingdom
9249,490123,84508B,NaN,184,2009-12-03 18:08:00,0.0,NaN,United Kingdom


In [62]:
zero_price.describe(include = "all")

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
count,1569.0,1569.0,468,1569.000000,1569,1569.000000,31.000000,1569
unique,1228.0,971.0,270,NaN,NaN,NaN,NaN,5
top,537534.0,22469.0,OWL DOORSTOP,NaN,NaN,NaN,NaN,United Kingdom
freq,57.0,9.0,8,NaN,NaN,NaN,NaN,1559
mean,NaN,NaN,NaN,114.505417,2010-06-15 02:29:39.732313,-87.031243,14204.806452,NaN
min,NaN,NaN,NaN,1.000000,2009-12-01 17:39:00,-53594.360000,12417.000000,NaN
25%,NaN,NaN,NaN,1.000000,2010-03-07 15:05:00,0.000000,12697.500000,NaN
50%,NaN,NaN,NaN,4.000000,2010-06-17 14:11:00,0.000000,14025.000000,NaN
75%,NaN,NaN,NaN,26.000000,2010-09-29 16:22:00,0.000000,14948.500000,NaN
max,NaN,NaN,NaN,10200.000000,2010-12-09 17:18:00,0.000000,18071.000000,NaN


In [63]:
zero_price["Description"].value_counts().head(20)

Description
OWL DOORSTOP                           8
POLYESTER FILLER PAD 45x45cm           7
PICNIC BASKET WICKER LARGE             7
FLAG OF ST GEORGE CAR FLAG             6
HEART OF WICKER SMALL                  6
AIRLINE BAG VINTAGE WORLD CHAMPION     6
POLYESTER FILLER PAD 40x40cm           5
SMALL POPCORN HOLDER                   5
WATERING CAN BLUE ELEPHANT             5
ENAMEL FIRE BUCKET CREAM               5
ENAMEL WASH BOWL CREAM                 5
PICNIC BASKET WICKER SMALL             5
IVORY KITCHEN SCALES                   5
DOOR MAT UNION JACK GUNS AND ROSES     4
temp                                   4
MILK PAN PINK RETROSPOT                4
RED RETROSPOT STORAGE JAR              4
HANGING METAL STAR LANTERN             4
AIRLINE BAG VINTAGE TOKYO 78           4
WATERING CAN PINK BUNNY                4
Name: count, dtype: int64

In [64]:
zero_price["Customer ID"].isnull().sum()

np.int64(1538)

In [65]:
zero_price["Invoice"].head(20)

3161     489659
3731     489781
4674     489825
5904     489861
6378     489882
6555     489898
6581     489903
6781     489998
7204     490015
9249     490123
10236    490150
14503    490543
15489    490688
15952    490716
16107    490727
17384    490758
18738    490961
18739    490961
20846    491053
21647    491106
Name: Invoice, dtype: object

# Investigation of Zero-Priced Transactions

A total of **1,569 transactions** had a unit price equal to zero.

## Investigation Findings

- Approximately 98% of these records do not contain a Customer ID.
- Many records have missing product descriptions.
- Some descriptions indicate internal operational activities.
- A small number of records appear to be promotional or complimentary customer items.

## Business Decision

The objective of this project is to analyse revenue-generating retail sales.

Since zero-priced transactions do not contribute to revenue and are predominantly internal or promotional records, they will be excluded from the cleaned analytical dataset.

This decision improves the accuracy of revenue analysis, customer analytics, and sales forecasting.

In [66]:
#Remove Zero Price Transactions

rows_before = len(clean_df)
clean_df = clean_df[clean_df["Price"]> 0]
rows_after = len(clean_df)

print(f"Rows before : {rows_before:,}")
print(f"Rows after : {rows_after:,}")
print(f"Rows removed : {rows_before - rows_after:,}")

Rows before : 513,134
Rows after : 511,565
Rows removed : 1,569


In [67]:
(clean_df["Price"] <= 0).sum()

np.int64(0)

# Duplicate Record Analysis

Duplicate records can artificially inflate business metrics such as revenue, sales quantity, and customer transactions.

Before removing duplicates, it is important to verify whether they are exact duplicates or represent legitimate repeated purchases.

Only exact duplicate records should be removed.

In [68]:
#Check for Duplicates Records

duplicate_count = clean_df.duplicated().sum()
print(f"Duplicate Record : {duplicate_count}")

Duplicate Record : 6835


In [69]:
duplicates = clean_df[clean_df.duplicated(keep=False)]
duplicates.head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
367,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
383,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom
384,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom


In [70]:
duplicates.sort_values(by=["Invoice","StockCode"]).head(30)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom


In [71]:
#Duplicates Percentage

duplicate_percentage = duplicate_count / len(clean_df) * 100

print(f"Duplicate Percentage: {duplicate_percentage:.2f}%")

Duplicate Percentage: 1.34%


# Duplicate Record Investigation

A total of **6,835 duplicate records** were identified, representing approximately **1.34%** of the cleaned dataset.

## Investigation Findings

The duplicate records were examined and found to be exact duplicates across all columns, including:

- Invoice Number
- Stock Code
- Product Description
- Quantity
- Invoice Date
- Unit Price
- Customer ID
- Country

Since all attributes are identical, these records represent duplicated data rather than genuine repeated customer purchases.

## Business Decision

The duplicate records will be removed to ensure accurate calculations of revenue, sales quantity, customer behaviour, and inventory metrics.

In [72]:
#Remove Exact Duplicate Records
row_before = len(clean_df)
clean_df = clean_df.drop_duplicates()
row_after = len(clean_df)

print(f"Rows before : {row_before:,}")
print(f"Rows after : {row_after:,}")
print(f"Rows removed : {row_before - row_after:,}")


Rows before : 511,565
Rows after : 504,730
Rows removed : 6,835


In [73]:
clean_df.duplicated().sum()

np.int64(0)

In [74]:
#Missing Values After Cleaning

missing_values = clean_df.isnull().sum()

missing_percentage = (clean_df.isnull().sum() / len(clean_df) * 100).round(2)

missing_report = pd.DataFrame({"missing Values": missing_values,"Percentage": missing_percentage})
missing_report

,missing Values,Percentage
Invoice,0,0.00
StockCode,0,0.00
Description,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Customer ID,103814,20.57
Country,0,0.00


# Missing Value Assessment After Data Cleaning

After removing cancelled transactions, inventory adjustments, zero-priced records, and duplicate rows, the dataset was re-evaluated for missing values.

Reassessing missing values is important because previous cleaning operations may have removed records that contained missing information.

The updated missing value report will guide the final preprocessing decisions before exploratory data analysis.

In [75]:
missing_report = (clean_df.isnull().sum().to_frame("Missing Values"))

missing_report["Percentage"] = (missing_report["Missing Values"] / len(clean_df) * 100).round(2)
missing_report

,Missing Values,Percentage
Invoice,0,0.00
StockCode,0,0.00
Description,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Customer ID,103814,20.57
Country,0,0.00


In [76]:
#Investigate Missing Description

missing_description = clean_df[clean_df["Description"].isnull()]

print("Missing Description Records:", len(missing_description))

missing_description.head(20)

Missing Description Records: 0


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country


In [77]:
missing_description["StockCode"].value_counts().head(20)

Series([], Name: count, dtype: int64)

In [78]:
#Dataset 1: Sales Analysis
sales_df = clean_df.copy()

In [79]:
#Dataset 2: Customer Analysis
customer_df = clean_df[clean_df["Customer ID"].notnull()].copy()

# Missing Customer ID Analysis

The cleaned dataset still contains transactions with missing Customer IDs.

## Business Interpretation

Customer ID is not required for:

- Revenue Analysis
- Product Performance
- Sales Trends
- Country-wise Sales
- Forecasting

However, Customer ID is essential for:

- Customer Segmentation
- RFM Analysis
- Customer Lifetime Value
- Repeat Purchase Analysis

## Business Decision

Instead of deleting all records with missing Customer IDs, two analytical datasets will be created:

1. Sales Dataset
   - Retains all valid sales transactions.
   - Used for revenue and product analytics.

2. Customer Dataset
   - Contains only transactions with valid Customer IDs.
   - Used for customer behaviour analysis.

In [81]:
#Sales Dataset Shape
print(sales_df.shape)

(504730, 8)


In [82]:
#Customers Dataset Shape
print(customer_df.shape)

(400916, 8)


In [83]:
#check for null values in customer dataset
customer_df.isnull().sum()

Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64

In [85]:
#Save both dataset to cleaned folder

import os

os.makedirs("./data/cleaned",exist_ok=True)

sales_df.to_csv("../data/cleaned/sales_data.csv",index=False)

customer_df.to_csv("../data/cleaned/customer_data.csv",index=False)

print("Datasets saved successfully!")

Datasets saved successfully!


## Why create two datasets instead of deleting missing Customer IDs?

A single cleaned dataset cannot optimally serve every business objective. Revenue and product analyses do not require customer identification, whereas customer analytics depends on it.

Creating separate datasets preserves all valid sales for operational analysis while ensuring customer-focused analyses are performed only on complete customer records. This approach minimizes unnecessary data loss and is commonly used in analytics projects.

### Data Cleaning Completed ###